# 04 - Real Streamlit Community Cloud Deployment

This chapter walks through full deployment process: repository setup, cloud linking, configuration, verification, and evidence capture.


## Section 1: What Is Streamlit Community Cloud?

### Definition
Streamlit Community Cloud is managed hosting platform for Streamlit applications with free tier.

### Theory
Cloud runner clones repository, installs dependencies, starts Streamlit using configured entrypoint.

### Motivation
Fast path from local prototype to public URL with minimal infrastructure setup.

### Benefits
- Fast onboarding.
- GitHub integration.
- Built-in logs and secret management.

### Limitations
- Resource constraints vs paid cloud.
- Best for demos, prototypes, lightweight production.

### Visual Explanation
![Cloud Workflow](../outputs/figures/cloud-workflow.png)

### Best Practices
- Keep startup lightweight.
- Cache expensive resources.
- Pin dependencies.

### Common Mistakes
- Missing `requirements.txt`.
- Wrong app entrypoint path.


In [1]:
from pathlib import Path
import subprocess

root = Path.cwd()
if not (root / "app.py").exists() and (root.parent / "app.py").exists():
    root = root.parent

checks = {
    "app.py": (root / "app.py").exists(),
    "requirements.txt": (root / "requirements.txt").exists(),
    ".streamlit/config.toml": (root / ".streamlit" / "config.toml").exists(),
    ".streamlit/secrets.toml.example": (root / ".streamlit" / "secrets.toml.example").exists(),
}
checks

gh_status = subprocess.run(["gh", "auth", "status"], capture_output=True, text=True, check=False)
print("gh auth status exit:", gh_status.returncode)
print((gh_status.stdout + gh_status.stderr).strip()[:500])


gh auth status exit: 0
github.com
  ✓ Logged in to github.com account pypi-ahmad (keyring)
  - Active account: true
  - Git operations protocol: https
  - Token: gho_************************************
  - Token scopes: 'gist', 'read:org', 'repo', 'workflow'


## Section 2: Step-by-Step Deployment Process

### Step 1 - Repository Creation
Create or use dedicated public repository for this project.

### Step 2 - Push Source
Push branch with tested deployment candidate to GitHub.

### Step 3 - Link in Streamlit Cloud
In Streamlit Cloud dashboard: `New app -> choose repo -> branch -> app path`.

### Step 4 - Configure Secrets
Add `HF_API_TOKEN` and optional `OLLAMA_BASE_URL` in cloud secrets panel.

### Step 5 - Deploy and Verify
Confirm app boots and each tab returns output.

### Step 6 - Capture Evidence
Save deployment URL, build log snippet, and screenshots.

**Example repository URL (replace with your own):**  
`https://github.com/<your-user>/<your-repo>`

If Streamlit Cloud blocks the flow with a sign-in challenge, finish login manually and continue deployment from the same repository.

### Code Explanation
Next cell writes deployment evidence template JSON for reproducible reporting.


In [2]:
from pathlib import Path
import json
from datetime import datetime, timezone
import os

root = Path.cwd()
if not (root / "outputs").exists() and (root.parent / "outputs").exists():
    root = root.parent

evidence = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "public_url": os.getenv("DEPLOYED_APP_URL", "NOT_SET"),
    "streamlit_build_status": os.getenv("STREAMLIT_BUILD_STATUS", "unknown"),
    "notes": "Populate env vars during/after real deployment run.",
}

out_dir = root / "outputs" / "deployment"
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / "deployment_evidence.json"
out_file.write_text(json.dumps(evidence, indent=2))
print("Wrote", out_file)
print(json.dumps(evidence, indent=2))


Wrote /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/MLOps, UI, and Deployment/Deploy Your AI App for Free in 3 Clicks/outputs/deployment/deployment_evidence.json
{
  "timestamp_utc": "2026-06-24T22:14:17.445953+00:00",
  "public_url": "NOT_SET",
  "streamlit_build_status": "unknown",
  "notes": "Populate env vars during/after real deployment run."
}


## Section 3: Troubleshooting Quick Hits During Deployment

### Common Failures
- `ModuleNotFoundError`: dependency missing in `requirements.txt`.
- App timeout on startup: heavy model initialization without caching.
- Build fails on wrong Python version.
- Secret missing (`HF_API_TOKEN`) causing inference errors.

### Best Practices
- Read build logs first.
- Reproduce same error locally.
- Fix root cause, then redeploy.

### Common Mistakes
- Re-deploying repeatedly without log analysis.
- Adding secrets to repository instead of cloud settings.
